In [8]:
import numpy as np
import pandas as pd
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import grangercausalitytests

# 1. Synthesize an interdependent two-variable economic system (e.g., Income & Consumption)
np.random.seed(42)
n_obs = 100

# Generating two stationary, mutually dependent series
e1 = np.random.normal(0, 1, n_obs)
e2 = np.random.normal(0, 1, n_obs)

y1 = np.zeros(n_obs)
y2 = np.zeros(n_obs)

for t in range(1, n_obs):
    y1[t] = 0.4 * y1[t-1] + 0.3 * y2[t-1] + e1[t]
    y2[t] = 0.2 * y1[t-1] + 0.5 * y2[t-1] + e2[t]

dates = pd.date_range(start='2026-01-01', periods=n_obs, freq='D')
df = pd.DataFrame({'Income': y1, 'Consumption': y2}, index=dates)

# Split into train and test sets (last 5 days withheld)
train_df = df.iloc[:-5]
test_df = df.iloc[-5:]

In [9]:
# 2. Initialize the VAR model on the training data
model = VAR(train_df)

In [10]:
# 3. Select the optimal lag order automatically using AIC
lag_order_selection = model.select_order(maxlags=5)
optimal_lag = lag_order_selection.selected_orders['aic']
print(f"--- Step 1: Order Optimization ---")
print(f"Optimal Lag Selection (AIC Criterion): {optimal_lag}\n")

--- Step 1: Order Optimization ---
Optimal Lag Selection (AIC Criterion): 1



In [11]:
# 4. Fit the optimal VAR model
results = model.fit(optimal_lag)

In [12]:
# 5. Execute a Granger Causality Audit
# Checking if 'Consumption' helps predict 'Income'
print("--- Step 2: Granger Causality Test Audit ---")
# maxlag matches our model parameters
gc_test = grangercausalitytests(train_df[['Income', 'Consumption']], maxlag=optimal_lag, verbose=False)
p_value = gc_test[optimal_lag][0]['ssr_ftest'][1]
print(f"Does Consumption Granger-cause Income? p-value: {p_value:.4f}")
print("If p-value < 0.05, the multivariate interaction is statistically significant.\n")

--- Step 2: Granger Causality Test Audit ---
Does Consumption Granger-cause Income? p-value: 0.0156
If p-value < 0.05, the multivariate interaction is statistically significant.



/usr/local/lib/python3.11/site-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(


In [13]:
# 6. Forecast into the future
# VAR requires the last 'optimal_lag' steps of historical data to kick off the forecast
forecast_input = train_df.values[-optimal_lag:]
forecast_values = results.forecast(y=forecast_input, steps=5)

# Wrap forecast in a clean DataFrame
forecast_df = pd.DataFrame(forecast_values, index=test_df.index, columns=df.columns)
print("--- Step 3: Combined System Future Forecast ---")
print(forecast_df.round(4))

--- Step 3: Combined System Future Forecast ---
            Income  Consumption
2026-04-06 -0.5724      -0.2899
2026-04-07 -0.3976      -0.1576
2026-04-08 -0.2960      -0.0742
2026-04-09 -0.2355      -0.0233
2026-04-10 -0.1991       0.0076
